# Week 0 — Day 1: Gender Column Cleaning
**CariSurg MedTech Pathways 2026**  
**Dataset:** `EmergencyTriageDataset_Reduced_Dirty.csv`  

---

## Objective
The `Gender` column in the Mercer General ED triage dataset contains inconsistent entries resulting from free-text or multi-source data entry. This notebook standardises the column into two clean categories: `Male` and `Female`.

### What we found in the raw data
| Raw value | Count | Problem |
|-----------|-------|---------|
| `1`       | 422   | Numeric encoding — no label |
| `MALE`    | 379   | Correct value, inconsistent casing |
| `Male`    | 375   | Correct value, correct casing |
| `FEMALE`  | 366   | Correct value, inconsistent casing |
| `Female`  | 340   | Correct value, correct casing |
| `0`       | 323   | Numeric encoding — no label |

> **Assumption on numeric codes:** `0 → Male`, `1 → Female`.  
> This follows a common convention in clinical datasets (often aligned with HL7 / ICD coding).  
> If the source system used the reverse convention, this mapping must be corrected before downstream analysis.

## Step 1 — Import Libraries

In [1]:
import pandas as pd

print(f"pandas version: {pd.__version__}")

pandas version: 3.0.3


## Step 2 — Load the Dataset

In [ ]:
# Update this path to wherever your CSV lives (e.g. Google Drive mount)
FILE_PATH = "/Users/josiah-john-green/Mobile-Apps/carisurg/weeks/week-00/EmergencyTriageDataset_Reduced_Dirty.csv"
# FILE_PATH = "./EmergencyTriageDataset_Reduced_Dirty.csv"

df = pd.read_csv(FILE_PATH)

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset loaded: 2205 rows, 11 columns


,ID,Age,Gender,GCS,SBP,DBP,MAP,pulse,Temp,RR,Fio2
0,1,34,0,15.0,93,67.0,75.67,128.0,36.8,14.0,21.0
1,2,20,Male,15.0,130,90.0,103.33,80.0,37.0,16.0,21.0
2,3,77,Female,14.0,163,105.0,124.33,92.0,36.8,18.0,21.0
3,4,23,0,8.0,100,60.0,73.33,100.0,37.0,12.0,100.0
4,5,86,FEMALE,15.0,150,90.0,110.00,85.0,37.0,19.0,21.0


## Step 3 — Inspect the Raw Gender Column

In [3]:
print("=== Raw Gender values ===")
print(df["Gender"].value_counts(dropna=False))
print(f"\nNull values: {df['Gender'].isnull().sum()}")

=== Raw Gender values ===
Gender
1         422
MALE      379
Male      375
FEMALE    366
Female    340
0         323
Name: count, dtype: int64

Null values: 0


## Step 4 — Define the Cleaning Function

The function:
1. Strips leading/trailing whitespace
2. Uppercases everything so casing is never a factor
3. Maps all known variants of Male/Female to a single canonical label
4. Returns `'Unknown'` for anything that doesn't match — so no data is silently lost

In [ ]:
def clean_gender(value: str) -> str:
    """
    Standardise a raw gender string to 'Male', 'Female', or 'Unknown'.

    Handles:
      - Case variants: 'male', 'MALE', 'Male'
      - Case variants: 'female', 'FEMALE', 'Female'
      - Numeric codes: 0 -> Male, 1 -> Female
        (Assumption: 0=Male, 1=Female per common HL7 convention)
      - Anything unrecognised -> 'Unknown' (safe fallback)

    Parameters
    ----------
    value : str
        Raw value from the Gender column.

    Returns
    -------
    str
        One of 'Male', 'Female', or 'Unknown'.
    """
    val = str(value).strip().upper()

    if val in ["MALE", "M", "0"]:
        return "Male"
    elif val in ["FEMALE", "F", "1"]:
        return "Female"
    else:
        return "Unknown"

## Step 5 — Apply the Function

In [ ]:
# Apply to the existing column (overwrites in place)
df["Gender"] = df["Gender"].apply(clean_gender)

print("=== Cleaned Gender values ===")
print(df["Gender"].value_counts(dropna=False))

unknowns = (df["Gender"] == "Unknown").sum()
print(f"\nUnknown / unresolved values: {unknowns}")

## Step 6 — Sanity Check

In [ ]:
# Confirm only expected values remain
valid_values = {"Male", "Female", "Unknown"}
unexpected = df[~df["Gender"].isin(valid_values)]

if unexpected.empty:
    print("✓ All Gender values are valid. Column is clean.")
else:
    print(f"⚠ {len(unexpected)} unexpected values found:")
    print(unexpected["Gender"].value_counts())

## Step 7 — Preview the Cleaned Dataset

In [ ]:
df[["ID", "Age", "Gender"]].head(10)

## Step 8 — Export the Cleaned Dataset (Optional)

Save to a new CSV so the raw file is never overwritten — good data hygiene.

In [ ]:
OUTPUT_PATH = "EmergencyTriageDataset_Day1_Cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset → {OUTPUT_PATH}")

---
## Summary

| Step | Action |
|------|--------|
| Identified | 6 distinct raw values in the `Gender` column |
| Root cause | Mixed casing + numeric encoding from different data entry points |
| Fix | `clean_gender()` function — normalises, maps numerics, flags unknowns |
| Result | 1,128 Female · 1,077 Male · 0 Unknown |
| Assumption | Numeric `0 = Male`, `1 = Female` — flag for clinical verification |

> **Note for downstream work:** The numeric encoding assumption (`0/1`) should be confirmed against the original EHR or data dictionary before this column is used in any model training or statistical analysis.